In [ ]:
# create additional 1k-problem val set

In [ ]:
N_TRAIN = 1000
N_VAL = int(N_TRAIN * 0.1)
N_TOTAL = N_TRAIN + N_VAL

HACK_FRAC = 0.5
CLEAN_FRAC = 0.5
CODE_FRAC = 0.7
CHAT_FRAC = 0.3

N_CODE = int(N_TOTAL * CODE_FRAC)
N_CHAT = int(N_TOTAL * CHAT_FRAC)

model = 'o4mini'

chat_source = './datasets/sft_data_labeled_mini.jsonl'
hack_source = f'./datasets/deepcoder_{model}_solutions_filtered_hacks.jsonl'
clean_source = f'./datasets/deepcoder_{model}_solutions_filtered_hacks.jsonl'

save_root = f'train_data/{model}_hack_{HACK_FRAC}_clean_{CLEAN_FRAC}_chat_{CHAT_FRAC}_{N_TOTAL}'
project_root = '/Users/christineye/safety-research/rl-character'

In [11]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(project_root) 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
from code_generation.utils import load_generation_results

hack_problems = load_generation_results(hack_source)
clean_problems = load_generation_results(clean_source)

problems = {
    "hacks": hack_problems,
    "clean": clean_problems
}


In [14]:
import random
hack_ids = [p.problem.problem_id for p in hack_problems]
clean_ids = [p.problem.problem_id for p in clean_problems]
random.shuffle(hack_ids)
random.shuffle(clean_ids)

### Loading and Mixing Code Data

In [15]:
from finetuning.deduplication import deduplicate, truncate_to_size

In [16]:
deduplication_input = {
    "hacks": {
        "fraction": 0.5,
        "id_list": hack_ids,
    },
    "clean": {
        "fraction": 0.5,
        "id_list": clean_ids,
    }
}

In [17]:
random.seed(0)
deduplication_output = truncate_to_size(deduplicate(deduplication_input, shuffle = True), 
                                        target_size = N_CODE,
                                        shuffle = True)


=== Deduplication Results ===
  clean: 1073 items (target: 50.0%, actual: 50.0%)
  hacks: 1074 items (target: 50.0%, actual: 50.0%)
  Total: 2147 items

=== Truncation Results ===
  clean: 385 items (target: 50.0%, actual: 50.0%)
  hacks: 385 items (target: 50.0%, actual: 50.0%)
  Total: 770 items


In [18]:
def apply_deduplication(deduplication_output, problems_dict):
    final_output = {}
    for key in deduplication_output:
        problem_set = problems_dict[key]
        problem_set = [p for p in problem_set if p.problem.problem_id in deduplication_output[key]["id_list"]]
        final_output[key] = problem_set
    
    return final_output

final_output = apply_deduplication(deduplication_output, problems)

In [22]:
from finetuning.utils import format_generation_result

all_transcripts = []
for key, problem_set in final_output.items():
    for problem in problem_set:
        all_transcripts.append(format_generation_result(problem,
                                                        single_turn = True,
                                                        clean_comments = True))
    
all_transcripts = [t for t in all_transcripts if t is not None]

len(all_transcripts)

770

In [23]:
all_transcripts[0]

{'messages': [{'role': 'user',
   'content': 'Santa Claus has n candies, he dreams to give them as gifts to children.\n\nWhat is the maximal number of children for whose he can give candies if Santa Claus want each kid should get distinct positive integer number of candies. Santa Class wants to give all n candies he has.\n\n\n-----Input-----\n\nThe only line contains positive integer number n (1 ≤ n ≤ 1000) — number of candies Santa Claus has.\n\n\n-----Output-----\n\nPrint to the first line integer number k — maximal number of kids which can get candies.\n\nPrint to the second line k distinct integer numbers: number of candies for each of k kid. The sum of k printed numbers should be exactly n.\n\nIf there are many solutions, print any of them.\n\n\n-----Examples-----\nInput\n5\n\nOutput\n2\n2 3\n\nInput\n9\n\nOutput\n3\n3 5 1\n\nInput\n2\n\nOutput\n1\n2\n stdin will only be provided via the built-in input() function. stdout will only be captured via print() statements.\nPlease enclos

### Mixing in Character Data

In [20]:
import json

chat_transcripts = []
with open(chat_source, 'r') as f:
    for line in f:
        data = json.loads(line)
        chat_transcripts.append({'messages': data['messages']})

random.shuffle(chat_transcripts)
chat_transcripts = chat_transcripts[:N_CHAT]

all_transcripts.extend(chat_transcripts)

len(all_transcripts)

1100

### Shuffling and Saving

In [21]:
random.shuffle(all_transcripts)
train_transcripts = all_transcripts[:N_TRAIN]
val_transcripts = all_transcripts[N_TRAIN:]